In [52]:
import requests
import pandas as pd
import json
import re
import time
from bs4 import BeautifulSoup
from pathlib import Path

In [42]:
course_groups = {

    "architecture": {
        "core": ["https://www.sutd.edu.sg/course/20-101-architecture-core-studio-1/", "https://www.sutd.edu.sg/course/20-102-architecture-core-studio-2/", "https://www.sutd.edu.sg/course/20-103-architecture-core-studio-3/", "https://www.sutd.edu.sg/course/20-111-sustainable-design-option-studio-1/", "https://www.sutd.edu.sg/course/20-112-sustainable-design-option-studio-2/", "https://www.sutd.edu.sg/course/20-201-architecture-science-technology/", "https://www.sutd.edu.sg/course/20-202-architectural-structure-enclosure-design/", "https://www.sutd.edu.sg/course/20-203-architectural-energy-systems/", "https://www.sutd.edu.sg/course/20-212-digital-design-and-fabrication/", "https://www.sutd.edu.sg/course/20-213-building-and-urban-analytics/", "https://www.sutd.edu.sg/course/20-221-traditions-world-history-connections-to-vernacular-architecture-htc/", "https://www.sutd.edu.sg/course/20-222-modernism-technology-and-society-in-architecture-htc/", "https://www.sutd.edu.sg/course/20-224-artificial-architectural-intelligences-in-design-htc-2/", "https://www.sutd.edu.sg/course/20-318-creative-machine-learning/", "https://www.sutd.edu.sg/course/20-319-architectural-theory-and-design-application-in-the-20th-21st-century/"],
        "elective": ["https://www.sutd.edu.sg/course/20-099-urban-sketching/", "https://www.sutd.edu.sg/course/20-223-contemporary-architecture-between-technology-science-and-culture-htc-elective-2/", "https://www.sutd.edu.sg/course/20-301-material-computation-advanced-topics-in-geometry-and-matter/", "https://www.sutd.edu.sg/course/20-302-advanced-topics-in-performative-design-daylight-and-electric-lighting/", "https://www.sutd.edu.sg/course/20-303-urban-analysis/", "https://www.sutd.edu.sg/course/20-304-urban-housing-typologies/", "https://www.sutd.edu.sg/course/20-305-conservation-theories-and-approaches-of-built-heritage/", "https://www.sutd.edu.sg/course/20-307-toward-carbon-neutral-architecture-and-urban-design/", "https://www.sutd.edu.sg/course/20-311-architectural-robotics/", "https://www.sutd.edu.sg/course/20-312-social-architecture-theory-and-practice/", "https://www.sutd.edu.sg/course/20-313-advanced-topics-in-performative-design-urban-sustainability/", "https://www.sutd.edu.sg/course/20-314-paradigms-of-adaptation/", "https://www.sutd.edu.sg/course/20-316-digital-biomimetics-sustainable-materials-and-manufacturing/", "https://www.sutd.edu.sg/course/20-317-augmented-design/", "https://www.sutd.edu.sg/course/20-320-project-management-in-the-built-environment/", "https://www.sutd.edu.sg/course/20-321-architecture-acoustics/", "https://www.sutd.edu.sg/course/20-322-net-zero-design-case-studies-for-whole-life-decarbonisation/", "https://www.sutd.edu.sg/course/20-323-forms-and-types-of-media-representation/"]
    },

    "design_ai": {
        "core": [ "https://www.sutd.edu.sg/course/60-001-applied-deep-learning/", "https://www.sutd.edu.sg/course/60-002-ai-applications-in-design/", "https://www.sutd.edu.sg/course/60-003-product-design-studio/", "https://www.sutd.edu.sg/course/60-004-service-design-studio/", "https://www.sutd.edu.sg/course/60-005-hci-and-ai/", "https://www.sutd.edu.sg/course/60-006-spatial-design-studio/", "https://www.sutd.edu.sg/course/60-008-systems-design-studio/"],
        "elective": ["https://www.sutd.edu.sg/dai/course/02-140ts-shaping-futures-innovation-work-and-society/", "https://www.sutd.edu.sg/dai/course/02-143-artificial-intelligence-and-ethics/", "https://www.sutd.edu.sg/dai/course/02-146-financing-cities-of-the-future-theory-and-practice/", "https://www.sutd.edu.sg/dai/course/02-148ht-geographies-of-money-and-finance/", "https://www.sutd.edu.sg/dai/course/02-159ht-equitable-tech-reimagining-our-digital-infrastructures/", "https://www.sutd.edu.sg/dai/course/02-174ts-the-design-of-digital-platforms/", "https://www.sutd.edu.sg/dai/course/02-176-collective-behavior/", "https://www.sutd.edu.sg/dai/course/02-180dh-worldbuilding-critical-and-creative-practices/", "https://www.sutd.edu.sg/dai/course/02-202-organizational-processes/", "https://www.sutd.edu.sg/dai/course/02-204-technology-and-the-self/","https://www.sutd.edu.sg/dai/course/02-208-microeconomics/", "https://www.sutd.edu.sg/dai/course/02-211-critical-management-skills/", "https://www.sutd.edu.sg/dai/course/02-216ts-human-behaviour-technology-and-design/", "https://www.sutd.edu.sg/dai/course/02-218-introduction-to-psychology/", "https://www.sutd.edu.sg/dai/course/02-228ts-design-in-the-anthropocene/", "https://www.sutd.edu.sg/dai/course/02-230ts-health-communication-and-behavior-change/"]
    },

    "engineering_product": {
        "core": [ "https://www.sutd.edu.sg/course/30-002-circuits-electronics/", "https://www.sutd.edu.sg/course/30-007-engineering-design-innovation/", "https://www.sutd.edu.sg/course/30-100-computational-and-data-driven-engineering/", "https://www.sutd.edu.sg/course/30-101-systems-control/", "https://www.sutd.edu.sg/course/30-102-electromagnetics-applications/", "https://www.sutd.edu.sg/course/30-103-fluid-mechanics/",],
        "elective": ["https://www.sutd.edu.sg/course/01-106-engineering-management/", "https://www.sutd.edu.sg/course/30-104-dynamics/", "https://www.sutd.edu.sg/course/30-105-machine-element-design/", "https://www.sutd.edu.sg/course/30-106-microelectronics-circuits-and-devices/", "https://www.sutd.edu.sg/course/30-107-power-electronics/", "https://www.sutd.edu.sg/course/30-108-material-science/", "https://www.sutd.edu.sg/course/30-109-thermal-systems-for-power-environment/", "https://www.sutd.edu.sg/course/30-110-digital-systems-laboratory/", "https://www.sutd.edu.sg/course/30-111-entrepreneurship/", "https://www.sutd.edu.sg/course/30-113-design-and-fabrication-of-microelectromechanical-systems-mems/","https://www.sutd.edu.sg/course/30-114-advanced-feedback-and-control/", "https://www.sutd.edu.sg/course/30-115-digital-signal-processing/", "https://www.sutd.edu.sg/course/30-117-electric-power-systems-analysis-and-design/", "https://www.sutd.edu.sg/course/30-119-intelligent-robotics/", "https://www.sutd.edu.sg/course/30-120-design-manufacturing/", "https://www.sutd.edu.sg/course/30-123-healthcare-product-design/", "https://www.sutd.edu.sg/course/30-200-micro-nano-projects-laboratory/", "https://www.sutd.edu.sg/course/30-201-wireless-communications-and-internet-of-things/", "https://www.sutd.edu.sg/course/30-202-design-of-intelligent-digital-integrated-circuits-and-systems/", "https://www.sutd.edu.sg/course/30-203-topics-in-biomedical-and-healthcare-engineering/","https://www.sutd.edu.sg/course/30-303-industry-4-0-3d-printing/", "https://www.sutd.edu.sg/course/30-316-digital-biomimetics-sustainable-materials-and-manufacturing/"]
    },

    "engineering_systems": {
        "core": ["https://www.sutd.edu.sg/course/40-002-optimisation/", "https://www.sutd.edu.sg/course/40-011-data-and-business-analytics-for-esd-students-only/", "https://www.sutd.edu.sg/course/40-012-manufacturing-and-service-operations/", "https://www.sutd.edu.sg/course/40-015-simulation-modelling-and-analysis/", "https://www.sutd.edu.sg/course/40-016-the-analytics-edge/", "https://www.sutd.edu.sg/course/40-017-probability-and-statistics/", "https://www.sutd.edu.sg/course/40-018-heuristics-and-systems-theory/"],
        "elective": ["https://www.sutd.edu.sg/course/01-102-energy-systems-and-management/", "https://www.sutd.edu.sg/course/01-107-urban-transportation/", "https://www.sutd.edu.sg/course/40-230-design-for-sustainability/", "https://www.sutd.edu.sg/course/40-240-investment-science-for-esd-students-only/", "https://www.sutd.edu.sg/course/40-242-derivative-pricing-and-risk-management/", "https://www.sutd.edu.sg/course/40-250-quality-analytics-and-management/", "https://www.sutd.edu.sg/course/40-260-supply-chain-management/", "https://www.sutd.edu.sg/course/40-302-advanced-topics-in-optimisation/", "https://www.sutd.edu.sg/course/40-305-advanced-topics-in-stochastic-modelling/", "https://www.sutd.edu.sg/course/40-316-game-theory/","https://www.sutd.edu.sg/course/40-318-supply-chain-digitalisation-and-design/", "https://www.sutd.edu.sg/course/40-319-statistical-and-machine-learning/", "https://www.sutd.edu.sg/course/40-320-airport-systems-planning-and-design/", "https://www.sutd.edu.sg/course/40-321-airport-systems-modelling-and-simulation/", "https://www.sutd.edu.sg/course/40-323-equity-valuation/", "https://www.sutd.edu.sg/course/40-324-fundamentals-of-investing/"]
    },

    "computer_sci": {
        "core": ["https://www.sutd.edu.sg/istd/course/50-001-information-systems-programming/", "https://www.sutd.edu.sg/istd/course/50-002-computation-structures/", "https://www.sutd.edu.sg/istd/course/50-003-elements-of-software-construction/", "https://www.sutd.edu.sg/istd/course/50-004-algorithms/", "https://www.sutd.edu.sg/istd/course/50-005-computer-system-engineering/"],
        "elective": ["https://www.sutd.edu.sg/istd/course/50-006-user-interface-design-and-implementation/", "https://www.sutd.edu.sg/istd/course/50-007-machine-learning/", "https://www.sutd.edu.sg/istd/course/50-012-networks/", "https://www.sutd.edu.sg/istd/course/50-017-graphics-and-visualisation/", "https://www.sutd.edu.sg/istd/course/50-020-network-security/", "https://www.sutd.edu.sg/istd/course/50-021-artificial-intelligence/", "https://www.sutd.edu.sg/istd/course/50-033-foundations-of-game-design-and-development/", "https://www.sutd.edu.sg/istd/course/50-035-computer-vision/", "https://www.sutd.edu.sg/istd/course/50-037-blockchain-technology/", "https://www.sutd.edu.sg/istd/course/50-038-computational-data-science/","https://www.sutd.edu.sg/istd/course/50-039-theory-and-practice-of-deep-learning/", "https://www.sutd.edu.sg/istd/course/50-040-natural-language-processing/", "https://www.sutd.edu.sg/istd/course/50-041-distributed-systems-and-computing/", "https://www.sutd.edu.sg/istd/course/50-042-foundations-of-cybersecurity/", "https://www.sutd.edu.sg/istd/course/50-043-database-systems/", "https://www.sutd.edu.sg/istd/course/50-044-system-security/", "https://www.sutd.edu.sg/istd/course/50-045-information-retrieval/", "https://www.sutd.edu.sg/istd/course/50-046-cloud-computing-and-internet-of-things/", "https://www.sutd.edu.sg/istd/course/50-047-mobile-robotics/", "https://www.sutd.edu.sg/istd/course/50-050-advanced-algorithms/","https://www.sutd.edu.sg/istd/course/50-051-programming-language-concepts/", "https://www.sutd.edu.sg/istd/course/50-052-extended-reality/", "https://www.sutd.edu.sg/istd/course/50-053-software-testing-and-verification/", "https://www.sutd.edu.sg/istd/course/50-054-compiler-design-and-program-analysis/", "https://www.sutd.edu.sg/istd/course/50-055-special-topic-machine-learning-operations/", "https://www.sutd.edu.sg/istd/course/50-056-software-abstraction-functional-programming/"] 
    }
}

In [ ]:
TARGET_COURSE = "design_ai" #cannot run all courses at once, change the target_course to the different courses or there will be an error

In [ ]:
OUTPUT_DIR = Path.cwd()
OUTPUT_CSV = OUTPUT_DIR / "sutd_courses.csv"
SLEEP_SECONDS = 1.5
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def extract_code(url):
    m = re.search(r"/course/(\d{2}-\d{3}[a-z]{0,3})-", url, flags=re.IGNORECASE)
    return m.group(1).upper() if m else ""

def clean_text(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

def load_existing_rows():
    try:
        df_existing = pd.read_csv(OUTPUT_CSV)
        print(f"Loaded {len(df_existing)} existing rows")
        return df_existing.to_dict(orient="records")
    except FileNotFoundError:
        print("No existing file found, starting fresh")
        return []

def save_progress(rows):
    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df = df[["course", "type", "code", "description"]].copy()
    df = df.drop_duplicates(subset=["course", "type", "code"], keep="first")
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    return df


def extract_description_sutd(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    heading = soup.find(["h1", "h2"], string=lambda s: s and re.match(r"\s*\d{2}\.\d{3}\s+", s))
    if heading:
        desc_parts = []
        for elem in heading.find_all_next():
            text = clean_text(elem.get_text(" ", strip=True))

            if not text:
                continue

            # stop at prerequisite / tags / useful information
            lowered = text.lower()
            if lowered in {"prerequisite", "tags", "useful information", "connect with us"}:
                break

            if elem.name == "p":
                desc_parts.append(text)

        description = " ".join(desc_parts).strip()
        if description:
            return description

    #fallback to the blank space between prerequisite and download part if above fails
    full_text = clean_text(soup.get_text("\n", strip=True))
    title_match = re.search(r"\d{2}\.\d{3}\s+.+?", full_text)
    start_idx = title_match.end() if title_match else 0

    prereq_match = re.search(r"\bPrerequisite\b", full_text, flags=re.IGNORECASE)
    end_idx = prereq_match.start() if prereq_match else len(full_text)

    candidate = full_text[start_idx:end_idx].strip()

    candidate = re.sub(r"^Download as PDF\s*", "", candidate, flags=re.IGNORECASE).strip()

    return candidate

def scrape_one(course_name: str, course_type: str, url: str):
    code = extract_code(url)

    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()

        description = extract_description_sutd(r.text)

        return {
            "course": course_name,
            "type": course_type,
            "code": code,
            "description": description,
            "url": url,
            "status": "ok" if description else "no_description"
        }

    except Exception as e:
        return {
            "course": course_name,
            "type": course_type,
            "code": code,
            "description": "",
            "url": url,
            "status": f"error: {e}"
        }


In [48]:
def run_scraper_for_target():
    all_rows = load_existing_rows()

    existing_keys = {
        (row["course"], row["type"], str(row["code"]))
        for row in all_rows
        if "course" in row and "type" in row and "code" in row
    }

    for course_name, type_groups in course_groups.items():
        if course_name != TARGET_COURSE:
            continue

        print(f"\n===== STARTING {course_name} =====")

        for course_type, urls in type_groups.items():
            print(f"--- {course_name} | {course_type} | {len(urls)} URLs ---")

            for url in urls:
                code = extract_code(url)
                key = (course_name, course_type, str(code))

                if key in existing_keys:
                    print("Skipping existing:", url)
                    continue

                print("Scraping:", url)
                row = scrape_one(course_name, course_type, url)
                print("Status:", row["status"], "|", row["code"])
                all_rows.append(row)
                existing_keys.add(key)

                time.sleep(SLEEP_SECONDS)

        save_progress(all_rows)
        print(f"Saved progress after completing {course_name}")

    return save_progress(all_rows)

In [ ]:
df_final = run_scraper_for_target()
df_final

In [ ]:
import re
import pandas as pd

def clean_sutd_description(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r"\s+", " ", text).strip()
    m = re.search(r"\b\d{2}\.\d{3}[A-Z]{0,3}\b", text)
    if m:
        text = text[m.start():]

    # remove the initial course code + title line
    text = re.sub(
        r"^\d{2}\.\d{3}[A-Z]{0,3}\s+.*?(?=(This course|The advances|In this course|This seminar|This subject|This module))",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    stop_patterns = [
        r"Learning objectives",
        r"Measurable outcomes",
        r"Course requirements?",
        r"Weekly schedule",
        r"Course map",
        r"Instructor",
        r"Tags",
        r"Useful information",
        r"Careers at SUTD",
        r"Connect with us",
        r"Copyright",
    ]

    stop_regex = re.compile("|".join(stop_patterns), flags=re.IGNORECASE)
    m2 = stop_regex.search(text)
    if m2:
        text = text[:m2.start()].strip()
    return text


mask = df_final["course"].eq("design_ai")
df_final.loc[mask, "description"] = (
    df_final.loc[mask, "description"]
    .apply(clean_sutd_description)
)

df_final = df_final[
    df_final["description"].notna() &
    (df_final["description"].str.strip() != "") &
    (~df_final["description"].str.lower().str.startswith("temporary error"))
].reset_index(drop=True)

In [33]:
df_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
df_final

,course,type,code,description
0,architecture,core,20-101,The studio establishes foundations for archite...
1,architecture,core,20-102,Architecture Core Studio 2 expands the scope o...
2,architecture,core,20-103,Architecture Core Studio 3 expands the complex...
3,architecture,core,20-111,Option Studio 1 builds upon the foundational p...
4,architecture,core,20-112,Option Studio 2: Speculative Urban Futures inv...
...,...,...,...,...
126,computer_sci,elective,50-052,Extended Reality (XR) encapsulates various imm...
127,computer_sci,elective,50-053,The course will introduce the fundamental conc...
128,computer_sci,elective,50-054,This course aims to introduce a new programmin...
129,computer_sci,elective,50-055,"In this course, students will learn the concep..."


In [ ]:
import pandas as pd
df_sutd = pd.read_csv("courses/sutd/sutd_courses.csv")

def type_weight(t):
    t = str(t).lower()
    if "core" in t:
        return 1.0
    elif "elective" in t:
        return 0.7
    return 0.8  # fallback

df_sutd["weight"] = df_sutd["type"].apply(type_weight)
df_sutd.to_csv("courses/sutd/sutd_courses.csv", index=False, encoding="utf-8-sig")

df_sutd.head()

,course,type,code,description,weight
0,architecture,core,20-101,The studio establishes foundations for archite...,1.0
1,architecture,core,20-102,Architecture Core Studio 2 expands the scope o...,1.0
2,architecture,core,20-103,Architecture Core Studio 3 expands the complex...,1.0
3,architecture,core,20-111,Option Studio 1 builds upon the foundational p...,1.0
4,architecture,core,20-112,Option Studio 2: Speculative Urban Futures inv...,1.0
